# SaaS Cohort Retention Analysis

**Dataset:** `user_events.csv`  
**Columns expected:** `user_id`, `event_date`, `event_type`

This notebook covers:
1. Environment setup and library imports
2. Data loading and basic validation
3. Data cleaning and type coercion
4. Cohort definition (first-event month per user)
5. Cohort matrix construction
6. Retention curve visualisation
7. Key findings and recommendations

## 1. Environment setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

sns.set_theme(style='whitegrid', palette='muted')

print('pandas  :', pd.__version__)
print('seaborn :', sns.__version__)

## 2. Load dataset and validate required columns

In [ ]:
df = pd.read_csv('user_events.csv')

print('Dataset shape:', df.shape)
print('\nColumn names:', df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Validate required columns are present
required_cols = ['user_id', 'event_date', 'event_type']
missing_cols  = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')
else:
    print('All required columns present:', required_cols)

print('\nData types (raw):')
print(df.dtypes)

print('\nNull counts:')
print(df.isnull().sum())

## 3. Clean and preprocess

In [ ]:
# Drop rows with nulls in required columns
before = len(df)
df = df.dropna(subset=required_cols)
print(f'Dropped {before - len(df)} rows with missing required values')

# Parse event_date to datetime
df['event_date'] = pd.to_datetime(df['event_date'], errors='coerce')

unparseable = df['event_date'].isnull().sum()
if unparseable:
    print(f'Warning: {unparseable} rows could not be parsed as dates — dropping')
    df = df.dropna(subset=['event_date'])

# Normalise string columns
df['user_id']    = df['user_id'].astype(str).str.strip()
df['event_type'] = df['event_type'].astype(str).str.strip().str.lower()

# Derive calendar month period (YYYY-MM)
df['event_month'] = df['event_date'].dt.to_period('M')

print('\nCleaned dataset shape:', df.shape)
print('\nEvent type distribution:')
print(df['event_type'].value_counts())

df.head()

## 4. Define cohorts

Each user's cohort is the calendar month of their **first recorded event**.

In [ ]:
# First event month per user
first_event = (
    df.groupby('user_id')['event_month']
    .min()
    .rename('cohort_month')
    .reset_index()
)

df = df.merge(first_event, on='user_id')

# Months since first event (cohort age)
df['cohort_index'] = (
    df['event_month'].dt.start_time.dt.to_period('M') -
    df['cohort_month'].dt.start_time.dt.to_period('M')
).apply(lambda x: x.n)

print('Cohort sizes:')
print(first_event['cohort_month'].value_counts().sort_index())

## 5. Cohort matrix — active users per cohort per month

In [ ]:
# Count distinct active users per cohort per cohort_index
cohort_data = (
    df.groupby(['cohort_month', 'cohort_index'])['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'active_users'})
)

# Pivot to matrix: rows = cohort month, columns = cohort age (months)
cohort_matrix = cohort_data.pivot_table(
    index='cohort_month',
    columns='cohort_index',
    values='active_users'
)

print('Cohort matrix (active user counts):')
cohort_matrix

In [ ]:
# Retention rate: divide each row by month-0 count
cohort_sizes  = cohort_matrix[0]
retention_pct = cohort_matrix.divide(cohort_sizes, axis=0) * 100

print('Retention rate matrix (%):')
retention_pct.round(1)

## 6. Visualisations

In [ ]:
# Heatmap of retention rates
fig, ax = plt.subplots(figsize=(14, 7))

sns.heatmap(
    retention_pct.round(1),
    annot=True,
    fmt='.1f',
    cmap='YlOrRd_r',
    linewidths=0.5,
    vmin=0, vmax=100,
    ax=ax
)

ax.set_title('Monthly Cohort Retention Rate (%)', fontsize=14, pad=12)
ax.set_xlabel('Months since first event')
ax.set_ylabel('Cohort (first event month)')
ax.set_yticklabels([str(p) for p in retention_pct.index], rotation=0)

plt.tight_layout()
plt.savefig('cohort_heatmap.png', dpi=150)
plt.show()
print('Saved: cohort_heatmap.png')

In [ ]:
# Retention curves — one line per cohort
fig, ax = plt.subplots(figsize=(12, 6))

for cohort, row in retention_pct.iterrows():
    row_clean = row.dropna()
    ax.plot(
        row_clean.index,
        row_clean.values,
        marker='o',
        markersize=4,
        linewidth=1.5,
        label=str(cohort)
    )

ax.set_title('Retention Curves by Cohort', fontsize=14)
ax.set_xlabel('Months since first event')
ax.set_ylabel('Retention rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.legend(title='Cohort', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('retention_curves.png', dpi=150)
plt.show()
print('Saved: retention_curves.png')

In [ ]:
# Average retention across all cohorts
avg_retention = retention_pct.mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(avg_retention.index, avg_retention.values, marker='o', linewidth=2, color='steelblue')
ax.fill_between(avg_retention.index, avg_retention.values, alpha=0.15, color='steelblue')

ax.set_title('Average Retention Curve (all cohorts)', fontsize=14)
ax.set_xlabel('Months since first event')
ax.set_ylabel('Average retention rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylim(0, 105)

for i, v in avg_retention.items():
    ax.annotate(f'{v:.1f}%', (i, v), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('avg_retention_curve.png', dpi=150)
plt.show()
print('Saved: avg_retention_curve.png')

## 7. Summary statistics

In [ ]:
print('=== Retention Summary ===')
print(f'Total users:          {df["user_id"].nunique()}')
print(f'Total events:         {len(df)}')
print(f'Date range:           {df["event_date"].min().date()} to {df["event_date"].max().date()}')
print(f'Number of cohorts:    {len(cohort_matrix)}')
print()
print('Average retention by month:')
for month, rate in avg_retention.items():
    print(f'  Month {month:2d}: {rate:.1f}%')

# Month-1 drop-off (key SaaS metric)
if 1 in avg_retention.index:
    m0 = avg_retention[0]
    m1 = avg_retention[1]
    print(f'\nMonth-0 to Month-1 drop-off: {m0 - m1:.1f} percentage points ({(m0 - m1) / m0 * 100:.1f}% of month-0 users lost)')